<a href="https://colab.research.google.com/github/ajtamayoh/In2Lab-ITM-at-IberLEF-NivELE-2026/blob/main/In2Lab_ITM_run_2_run_3_run_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In2Lab-ITM at IberLEF NivELE 2026: Automatic CEFR Level Classification for Spanish Learner Texts: A Comparative Study of a Pretrained LLM and Prompt Engineering

Antonio Jesus Tamayo Herrera (University of Antioquia, UdeA)

Juan Felipe Zuluaga Molina (Institución Universitaria ITM)

Correspondence: antonio.tamayo@udea.edu.co

# 1. Instalación de Librerías y Configuración

Primero, instalamos el SDK de OpenAI y preparamos las credenciales.

In [ ]:
!pip install openai pandas scikit-learn

import pandas as pd
import json
import os
from openai import OpenAI
from sklearn.metrics import f1_score, recall_score, precision_score, classification_report

# Configuración de tu API Key de OpenAI
os.environ["OPENAI_API_KEY"] = ""
client = OpenAI()

# 2. Carga y Preparación de Datos

Este bloque lee el CSV.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. Cargar el corpus (CSV)

#Datos de test de NivELE
df_final = pd.read_csv('/content/drive/MyDrive/data/test.csv') # columnas: id, text

print(f"Total de registros cargados: {len(df_final)}")
df_final.head()

# 3. Función de Clasificación con GPT-4o Mini

Definimos la lógica para consultar al modelo. Usamos un System Prompt estricto para que solo responda con la categoría del nivel de lengua, lo cual facilita el procesamiento.

In [ ]:
# Definimos los ejemplos para el Few-Shot

# Examples for Run 2

few_shot_examples = [
    {"text": """Como estais ? ! Hola amigos ! ? Espero que todo esta bien ! ? Como eran vuestros vacaciones ? Este verano yo ha viajado mucho por el Europa . Era mi primera vez volando en el avion y cuando pensaba de este , me daba miedo . ! Pero pasaba mucho mejor que esperaba ! Ha ido con mi familia y amigos de mis padres . Hemos visitado Italia y Espana ! Fue increible ! Primer pais era Italia , alli hemos comido unos platos tipicos de Italia , eran ricisimos . Ademas , hemos visitado muchas museas diferentess y lugares historicas .""", "label": "A1"},
    {"text": """¿ Que tal ? un beso Espero que todo va bien por tu . Mi padre Rene esta mejor , no le duele mas la pierna Es pronto el compleaño de mi hermano , ¡ 30 30+años ! madre mia vamos a hacer una fiesta y ! que fiesta ! Mi madre , Janette , trabaja siempre en el clinica , pero la clinica cierrada a el fin de esta año . Juntos viven siempre en el pueblo . Nadine""", "label": "A1"},
    {"text": """¿ Cómo estas chica ? Había regresado de México con mi madre . Fue muy divertido y en México había mucha sol . Mi madre y yo pasabamos mucho tiempo en la playa . ¡ Quiero regresar pronto ! La gente fue muy simpatica y pude practicar hablar en espaÑol casi todo el tiempo . Todos me ayudaron cuando no conocí las palabras si pude explicar las en íngles . Aprendía mucho durante mis vacaciones . También aprendí de los Mayans y todo que hayan en México . Ellos estaban muy moderna y inteligente .""", "label": "A1"},
    {"text": """Hola querida Sara , espero que estes muy bien y que todo en el trabajo vaya como tu quieras . No te vi en la universidad toda la semana y la verdad es que perdi tu numero de teléfono como muchos otros . Te estoy mandando este correo electrónico para pedir te una favor : es que la semana pasada , cuando iba a Marrakech para visitar a mi familia , tome mis notas de todos los cursos conmigo para trabajar un_poco , y en el camino de regreso , perdí mi equipaje en el avion . Es una situacion tan dificil y estoy muy avergonzada por pedir te eso pero eres la única persona con la que puedo contar . ¿ Puedes por_favor prestar me tus notas para fotocopiar las ? Esto me ayudaré mucho , no te puedes imaginar . Me puedes mandar un correo electrónico para decir me cuando y donde nos vemos ( prefiero que sea en la universidad ) o llamar me por teléfono ( 05554893 ) . Hasna .""", "label": "A2"},
    {"text": """Cuando llegué a el destino , le propuso que si se podía viajar en Tokyo juntos , y ella me dijo que sí . ¡ Era un momento hermoso para mí ! Nosotros paseamos por las calles en Shinjuku ( Un lugar en Tokyo ) , compramos unas cosas en las tiendas y tomamos varios tipos de las comidas japonesas . Nos discutimos mucho y me llevé muy bien con ella . Pero el tiempo pasaba rápidamente y teníamos que despedir nos . Creo que me divertí mucho , aunque estos días he estado todos los días pensando en ella . Antes no me encontraba con tanta buena chica como ella . Tengo su teléfono pero no quería llamar la . No me gusta el sentimiento triste . Espero tus respuestas . Un abrazo FRANCISCO""", "label": "A2"},
    {"text": """Querida Anita , ¡ Yo estoy muy contenta ! El ultimo año hice muchas cosas diferentes y buenas en mis vacaciones , quiero contar te . Yo fui con mi hermano a Egito y conoci las pyramides , la cultura , la comida , la música , la religion , las personas y el Nilo . Estuve en un barco por una semana en el Nilo , el sol me despiertó todos los días con su magía . Fui en el Vale_de_los_Renes , donde eston las múmias de muchas dinastias . Hacia más de 40 grados y sin árboles ... ¡ pero todo estaba perfecto ! En Cairo conoci su famoso museo , es muy grande y tiene muchas cosas para aprender . ¿ Y tus vacaciones como fueron ? Me conta . Hasta lluego , Maria_Alcina .""", "label": "A2"},
    {"text": """Hola Carlos ! Hace mucho que no te veo hombre ! Cómo está tu vida ? Trabajando muchísimo como siempre , no ? Continuas en el Instituto_Cervantes ahí de España ? Espero que sí , pues tu trabajo es buenísimo . Y por_favor , dí me lo ! Todavía vives con Maria ? ya estáis casados o sigues soltero ? Yo sigo soltera como siempre , tu sabes , hay cosas que no cambian . Pero tengo que contar te lo de Cláudia , está casada con Francisco , te acuerdas de Francisco ? El chico rubio de nuestra clase , el que no hablaba con nadie . Pues esta casada con el , y ella también ya no habla con nadie . Intento llamar la por teléfono pero sin respuestas . Es una lástima pues erámos muy amigas . Ya sabes , Carlos , que voy a viajar a Europa el próximo año . Voy a quedar me en Francia estudiando por seis meses . Después de esto tendré un més para quedar me donde quiera , por eso he pensado en visitar te . Sabes cuanto me gustaría conocer Madrid y , además , te echo de menos , pues hace mucho tiempo que no vuelves a el Brasil .""", "label": "B1"},
    {"text": """Querida Carla , Quiero preguntar por tu y de tu familia Como estas ? Como esta tus padres y tu pequeno hermano , Hugo ? Y tu trabajo ? Eras siempre una estudiante o has encontrado un trabajo ? Por_favor . Puedes preguntar a tu novio si puedes prestar me un coche de su sociedad ? Es simple para un o dos meses . Me haces falta , creas un vacio interior en mi y me siento sola sin tu cerca de mi porque eras mi amiga preferida . Si quieres que haga una cosa para ti , di me , una ayuda en tu trabajo de clase , o profesional , si quieres que me ocupe de tu gato o que pregunte por una persona . Claro , si tienes el deseo de pasar un momento en mi casa , pregunta me . Todo lo que quieres mi amiga ! Pero , tengo una cosa a preguntar te , ayer , he perdido mi coche porque una persona me la ha robada . Puedes contestar a mi carta enviando una otra carta . Vivo a el 42 Rue_Sanche_de_Pomiers , 33000 BORDEAUX .""", "label": "B1"},
    {"text": """En carnaval de ese año ( 2011 ) Bruno mi amigo de clase , fue para Olinda y bien estava lá conm su novia , sus amigos y estava muy feliz y bebendo mucho como de costume hahahahaha . Pero su día no fue tan bueno como estoy hablando , o que ocorrió fue que otro hombre se quedó enamorado de él y estava cerca de Bruno todo lo tiempo , y él no percebió nada a princípio . Quería tirar lo hombre y matá lo tambíen , pero sus amigos y su novia no dejaran ! Bruno perdió su término de carnaval , se fue para su casa y ponió a durmir y olvidar de el ocorrido ! ! ! Pero un otro amigo ( que se llama Oseas ) , también tuvo una experiencia similar , fue quando yo y él fuimos a un ' show ' en Chevrolet_Hall , bien havia uns diez hombres ( ' gays ' ) cerca de un couche y nosotros estávamos cerca de lo mismo couche ... cuando vimos , los ' gays ' estavan a llamar mi amigo , pensavan que él también era de el mismo club kkkkkkkkkkkkkkkk . Fue muy " diferente " esa experiencia que mi amigo tuve ! Y fue también muy similar a história de Bruno en carnaval .""", "label": "B1"},
    {"text": """Y voy a graduar de mi universidad en dos años . ¿ Hay algunas posibilidades para obtener la residencia universaria ? ¡ Gracias por su respuesta de_antemano ! En_cuanto_a mí , soy estudiante de la facultad de traducción e interpretación de la Universidad_Estatal_Lingüística_de_Moscú , como yo he dicho . Me gusta estusdiar mucho y aprender muchas cosas nuevas y eso es la primera razón porque quiero seguir estudiando y , por_supuesto , ir a a el país de la lengua que estoy aprendiendo . Soy una persona muy trabajadora , y me gusta hacer el trabajo de principio a fin , sin posponer todo para más tarde . Siempre quiero hacer todo a el más alto nivel ya_que es fundamental en cada trabajo - Entonces , encuentro fácilmente el contacto con la gente , y me parece que eso es muy relevante en mi profesión . Entonce , voy a despedir me . Es un placer a escribir y tratar a entrar en vuestro Universidad . Y me espero mucho que va a contactar me .""", "label": "B2"},
    {"text": """Lev_Zykov . Un saludo , me llamo Lev_Zykov y me interesa su programa de posgrado de lengua y literatura española , porque me interesan los estilos en los que la literatura se ha escrito , también me gustaría saber más la literatura española y la lengua española , porque hay mucho que todavía no se en esta dirección , por_lo_tanto , me interesan las fechas cuando este programa se empieza y cuando se termina , también me gustaría saber condiciones de admisión , cuanto cuesta la participación en el programa y si el precio de alojamiento está incluido en el precio de el programa . Tengo 7 publicaciónes en ingenieria y un patento , pero me gustaría saber más en filología y literatura . Soy activo , me interesan muchas cosas de este mundo , especialmente cuando los explican bueno , también me gusta la cultura española y latinoamericana , yo bailo el tango argentino y me gusta mucho el arte . ¡ Muchas gracias por la atención !""", "label": "B2"},
    {"text": """El cigarro es uma droga que contiene muchas substancias que causan vicio y adicción a las personas . Por ser una droga lícita , hay un gran numero de personas que fuman y muchas vezes ni estan preocupada con su salud ni con la salud de las otras persona . En muchos lugares de el mundo existem reglas a cerca_de la utilizacíon de el cigarro . Una de estas reglas se traduz en poder fumar en lugares publicos o no . En mi opinión , no deveria ser permitido que las personas fumasen en lugares publicos , pues se aprobarmos la liberacíon de el uso de el cigarro en publico , estaremos ayudando , a estas personas a hacer daño a ellas mismas y aún a las personas que circulan en estes ambientes comunes . El humo producido por el cigarro puede causar cancer de pulmón , de lengua , de laringe , además acarretar daños a el corazón , desgatar los dientes , producir substancias inflamatorias en los vasos sanguineos , depresión , abuso de otras drogras , etc. Diante_de tantos problemas y maleficios que el cigarro puede causar , hago la seguiente pregunta : Por que las personas fuman ? Será que tienem consciencia de las consequencias de el uso de el cigarro ? Será que no imaginan que estan haciendo daño a si mismas y tambien a las otras personas con que conviven ? Concluo esta mi opinión , decindo que para mi , no debe ser permitido fumar en lugares publicos , porque ademas de hacer daño a los individuos en general , también estaremos haciendo daño a el medio_ambiente . Intenten cambiar este habito por otro saludable ! Se preocupen no solamente con sus vidas , pero tambien con la de las otras personas , con la naturaleza ! Fumar es un tipo de suicidio asistido , digan no a esta adiccíon !""", "label": "B2"},
    {"text": """Acudí a su compañía la dirección de la cual ví en una página web y personalmente llegué a su oficina para asugurar me de que tenían todos los documentos necesarios y que la compañía estaba acreditada . Allí una empleada suya me contó de las tarifas , me mostró unos cuantos comentarios de su clientela , que de_hecho eran más que buenos , y me aseguró que no iba a tener nigunos problemas con gas y electricidad una vez firmado el acuerdo con su compañia . Los precios me parecieron un_poco más altos de lo normal pero con la evaluación tan positiva de su empleada pensé que valía la pena probar lo . La primera semana todo funcionaba sin contratiempos : la luz se encendía y el gas tampoco fallaba , no me podía quejar . Todo comenzó el 18_de_noviembre . Fue entonces cuando se me cortaron las luces por primera vez . Pensé que era un incidente aislado y que no volvería a pasar , pero lo mismo pasaba una y otra vez . En total he tenido 5 cortes de electrecidad en un mes ! No sé si es una practica común para su compañía pero yo me niego a pagar tanto dinero en vano ! Y encima ayer recibí la factura y por alguna razón las cifras que ví no coincidían con las que me habían prometido . Y no esperen que lo quede así ! Si su compañía no arregla el funcionamiento de la electrecidad y de el gas y si no me paga la indemnización dentro_de una semana , me veré obligado a acudir a las autoridades e incluso a el corte si es necesario . Espero que no tarden en responder . Con respeto , Ramón_Vidal""", "label": "C1"},
    {"text": """Estimado Sres. , Me llamo Nicolas_Berhorst , economista , vivo en el barrio norte de la ciudad de Rosario hace 20 años . Escribo este correo pues estoy harto de aguardar por el teléfono oyendo aquella terrible música de ascensor en la cual tengo que escuchar a cada vez que me pasan por todos los empleados de su empresa , pero nadie puede resolver a mi problema . ¡ Lo que pasa es que así_que nuestra presidente privatizó esta empresa , y ustedes tomaron su administración , la cuenta llegó as las nubes ! ¡ Antes yo pagaba 50 pesos por mes en la cuenta de energía , y ahora la misma subió para 200 pesos ! ¡ Si uno hace las cuentas , es 300 % de aumento ! Santa_Fé entera debe estar mordiéndo se de rabia ahora . Y yo no me recuerdo que las aguas de el Rio_Paraná secaron de un mes para acá . Entonces me parece que los precios no pueden subir todo eso . ¿ Yo imagino como será en el invierno cuando tendremos que encender la calefacción , como vamos hacer para pagar ? ¡ Lo que me deja mas loco que una cabra es que cuando supimos de ese cambio , nos fue prometido que los precios iban a subir un_poco , pero no 300 % ! ¡ Yo fue engañado , y sé seguramente que no soy el único ! Es como comprar un producto que no te sirve porque es de baja calidad . ¿ Pregunto a usted , te parece justo cobrar un absurdo de un precio por La energía y córta la por 5 días en un único mes ? Es 17 % de el precio que no debía ser cobrado de nosotros . ¡ Es una calamidad !""", "label": "C1"},
    {"text": """Mi nombre es Ana_María_Gómez_Martínez . Le escribo para quejar me de la calidad de servicios que me proporciona su compañía . Tengo todas las razones para estar muy discontenta . Y no conozco ninguna circunstancia que que sirva de excusa para su incapacidad de hacer su trabajo como se debe . Para empezar , la primera factura que he recibido sea mucho más alta que la provista . Hace un mes su empleado me prometió por teléfono que sería 50 euros y el pago que debo efectuar en realidad es 77 euros . ¿ No les parece sorprendiente ? 103 Podría consentir se lo , si el sericio fuera perfecta . Pero es ni siquiera satisfactorio . Me indigna el hecho de que de los 30 días de el mes haya tenido una vida tranquila solo durante los 25 debido_a los cortes de electricidad que han tenido en otros 5 . Primero , casi todos los productos en mi nevera no servieron para nada tras el corte más largo .""", "label": "C1"}
]

In [ ]:
# Examples for Run 3

few_shot_examples = [
    {"text": """¡Hola!
¿Qué tal?

Me llamo Carlos.
Soy italiano, de Firenze.
Vivo en Madrid.
Soy estudiante.
Tengo 19 años.

Soy alto, tengo el pelo negro, corto y liso.

En mi tiempo libre juego al fútbol, escucho música, voy al cine y salgo con mis amigos.
Me gusta mucho viajar, escuchar música y comer.

Mi ciudad es famosa por la comida.

Me gusta estudiar español también, porque España es muy interesante y bonita.

No me gusta mucho estudiar por la noche.

Gracias.
¡Adiós!""", "label": "A1"},

    {"text": """¡Hola!

Mis últimas vacaciones fueron en verano.
Mi familia y yo viajamos a Tailandia por dos semanas.

Casi todos los días fuimos a la playa cerca del hotel y nadamos en el mar. También tuvimos varias excursiones,por ejemplo estuvimos en templo "Wat Chalong".
Es muy grande y con una vista muy bonita.
Está situado en la montaña.

Por otro día estuvimos en la excursión del mercado flotante.
La verdad,estuvo un poco extraño pero interesante.

El lugar más memorable para mi familia y yo fue la isla "Phi Phi".
Este lugar con una playa muy hermosa que tiene agua muy clara y arena blanca hizo relajarnos mucho.

Mi familia y yo tenemos las impresiones muy bonitas de Tailandia!
Este país no se puede comparar con otros porque es muy diferente.

Creo que nuestras próximas vacaciones estarán en Italia porque tenemos muchas ganas de visitarlo.

Con mucho amor,Ana.""", "label": "A2"},

    {"text": """Estimados señores:

Me llamo Natalia.
Soy clienta de su compania desde hace muchos años y nunca he tenido ningun problema antes.

Pero 3 semanas antes de 15 septiembre de 2018 su compania perdio mi equipaje cuando yo regresaba a Barcelona.

Yo hice una reclamacion en el aeropuerto,pero su compañia no quiso tomar ninguna responsabilidad.
Hasta ahora no he recibido ni mi maleta,ni ninguna informacion sobre donde esta.

Espero que haya pasado algun error y puedan ayudarme solucionar este problema.

Mi maleta era de color negro,mediana,90 l,de marca American Tourister.
Adentro habia muchas cosas importantes para mi,por ejemplo ropa,documentos y regalos para mi familia.

El completo listo de cosas pueden encontrar en mi reclamacion de 15 septiembre de 2018.

Por si a caso pueden poner en contacto conmigo por mi correo electronico o por telefono.

Espero que me respondan lo mas pronto posible.

Su clienta,
Natalia""", "label": "B1"},
    {"text": """Estimados señores de la dirección de la universidad:

Me llamo Laura Martinez y tengo 23 años.
He estudiado filología española en la universidad de Sevilla y terminé mis estudios hace dos meses.

Las lenguas siempre han sido algo que me interesan mucho desde mi infancia y no podría imaginar estudiar otra cosa diferente.

Como mis estudios de grado ya han terminado, me gustaría mucho continuar mi formación con un postgrado y el programa de lengua y literatura española de su universidad en Chile me parece muy interesante.

Durante mis estudios de grado hice prácticas en un instituto español y también escribí un pequeño trabajo sobre el uso del español entre los jovenes y los cambios que aparecen en la lengua hoy en día.

Lo que me gustaría hacer despues del postgrado es trabajar como profesora en un instituto porque enseñar es algo muy interesante para mi.

Sería muy bueno si me podría dar algunas informaciones sobre el programa de postgrado.

Primero, no estoy segura sobre la duración del programa.
Despues, me gustaría saber si existen algunas condiciones de admisión, por ejemplo experiencia profesional o algunas notas especiales en la universidad.
Tambien me interesaría recibir informacion sobre la organización del programa porque vivo actualmente en España y tengo que preparar muchas cosas antes de ir a Chile.

Otra cosa que me preocupa es el coste del programa.
Me interesaría también saber si hay posibilidad de becas o residencia universitaria porque mi familia tiene que gastar mucho dinero para mis estudios y una ayuda financiera sería muy importante para mi.

Finalmente, mi última pregunta es sobre los servicios que ofrece la universidad para los estudiantes extranjeros.

Creo que este programa de postgrado sería perfecto para mi porque la lengua y la literatura española me interesan mucho.

He recibido dos cartas de recomendación de mis profesores en la universidad y mi nota final de grado es 8,7.

Soy una persona muy trabajadora y responsable que intenta siempre hacer las cosas lo mejor posible cuando quiere conseguir algo.

Quiero agradecerles mucho por su atención.
Espero su respuesta pronto.

Atentamente,
Laura""", "label": "B2"},

     {"text": """Estimados señores:

Me llamo Carlos Mendez y vivo en la calle Mayor 27, Alcázar de San Juan, Ciudad Real.
Les escribo para comentar la situación que he experimentado con su compañia durante las últimas semanas.

El día 3 de abril firmé un contrato con su empresa para recibir los servicios de gas y electricidad en mi vivienda.
Sin embargo, desde el principio han aparecido varios problemas que no esperaba encontrar.

En primer lugar, los datos que habiamos acordado por telefono no coinciden con los que aparecen en la factura que recibi hace unos días.
Me sorprendió bastante porque durante nuestra conversación todo parecia claro y bien explicado.

Además, cuando intenté aclarar la situación llamando a su servicio de atención al cliente, la persona que me atendió no me dio muchas explicaciones y la impresión que tuve fue que queria terminar la conversación lo antes posible.
Incluso me dijo que probablemente era un error del sistema pero no me explicó que debía hacer en este caso.

Otro problema importante es que durante este corto periodo he tenido varios cortes de electricidad.
En total han sido aproximadamente cuatro días en los que no he tenido luz durante varias horas seguidas.
Despues la electricidad vuelve a funcionar normalmente, pero al cabo de un tiempo vuelve a pasar lo mismo.

La situación resulta bastante complicada para mi familia porque tengo dos hijos pequeños y sin electricidad ni gas es dificil organizar la vida diaria de manera normal.
Por ejemplo, algunas tardes hemos tenido que estar en casa sin luz y preparar la comida se vuelve bastante complicado.

Por todo esto, me gustaría pedirles que revisen mi contrato y las condiciones del servicio.
Creo que sería necesario comprobar los precios que aparecen en la factura porque no corresponden con lo que habiamos acordado anteriormente.

Tambien agradecería que pudieran informarme sobre el motivo de los cortes de electricidad y si existe alguna solución para evitar que sigan ocurriendo en el futuro.

No quiero exagerar la situación, pero sinceramente esperaba un servicio más estable por parte de su empresa.

Espero que puedan revisar mi caso y darme una respuesta lo antes posible.

Atentamente,
Carlos""", "label": "C1"}
]

In [ ]:
#prompt para run 2 y run 3
# “role”: “system”, “content”: “”“You are an expert in assessing language proficiency according to the CEFR for students of Spanish as a foreign language. Assign only one of the following categories: “A1”, “A2”, “B1”, “B2”, or “C1”. Write the assigned category in all caps. Do not explain your decision.

def classify_nivele_few_shot(text):
    # Construimos el prompt con el patrón few-shot
    messages = [
        {"role": "system", "content": """Eres un experto clasificador de nivel de lengua usando CERF, para aprendices del español como lengua extranjera. Asigna únicamente con una de las siguientes categorías: "A1", "A2", "B1", "B2" or "C1". Considera las letras mayúsculas para escribir la cateogría asignada. No expliques tu decisión.

        Instrucciones:

        Rasgo lingüístico | Indicador CEFR | Qué observar | Feature computacional

        Tiempos verbales | A1–A2: presente / B1+: pasados / B2+: variedad | Uso de pasado, subjuntivo, condicional | Frecuencia por tiempo verbal (conteo y proporción)
        Variedad verbal | A1: limitado / B2+: amplio repertorio | Cantidad de formas verbales distintas | Número de formas verbales únicas
        Subjuntivo | B1–B2 en adelante | Presencia y uso correcto | % de verbos en subjuntivo
        Longitud de oración | A1: corta / B2+: más larga | Número de palabras por oración | Promedio de palabras por oración
        Subordinación | B1+: subordinadas / B2+: complejas | Uso de "que", "aunque", "cuando" | Número de subordinadas por oración
        Profundidad sintáctica | B2–C1: estructuras anidadas | Oraciones con múltiples niveles | Profundidad del árbol sintáctico
        Conectores básicos | A1–A2: y, pero, porque | Uso limitado | Conteo de conectores básicos
        Conectores avanzados | B2+: sin embargo, por lo tanto | Variedad discursiva | Conteo y diversidad de conectores
        Diversidad léxica | A1: baja / B2+: alta | Repetición vs variedad | Type-Token Ratio (TTR)
        Léxico sofisticado | B2–C1 | Uso de palabras abstractas o poco frecuentes | Frecuencia de palabras raras (según corpus)
        Errores gramaticales | A1–B1: frecuentes | Concordancia, género, verbos | Errores por cada 100 palabras
        Tipos de error | Varían por nivel | Tipo de error dominante | Clasificación de errores (género, tiempo, etc.)
        Cohesión | B1+: uso de referencia | Pronombres, sustituciones | Frecuencia de pronombres y anáforas
        Repetición léxica | A1–A2: alta | Repetición de palabras | Ratio de repetición léxica
        Estructura discursiva | B2+: organización clara | Introducción, desarrollo, cierre | Detección de estructura (heurística o modelo)
        Longitud del texto | Generalmente crece con nivel | Número total de palabras | Conteo total de tokens
        Densidad léxica | B2+: mayor densidad | Más contenido que función | Ratio palabras léxicas / totales
        Complejidad global | Aumenta con nivel | Mezcla de factores | Índices combinados (ej: complejidad sintáctica + léxica)

        """}
    ]

    # Añadimos los ejemplos al historial (Few-Shot)
    for example in few_shot_examples:
        messages.append({"role": "user", "content": example["text"]})
        messages.append({"role": "assistant", "content": example["label"]})

    # Añadimos el texto real a clasificar
    messages.append({"role": "user", "content": text})

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0,
            max_tokens=2 # Subimos a 2 por si el tokenizador separa el label de un espacio
        )
        return response.choices[0].message.content.strip().lower()
    except Exception as e:
        print(f"Error: {e}")
        return "error"

In [ ]:
from openai import RateLimitError
import time

def safe_call(func, *args, **kwargs):
    delay = 0.005  # 1 ms inicial

    while True:
        try:
            return func(*args, **kwargs)

        except RateLimitError:
            time.sleep(delay)
            delay = min(delay * 2, 1)  # backoff exponencial hasta 1 segundo

        except Exception as e:
            # opcional: otros errores los lanzas
            raise e

# Procesamiento por lotes para training - Corpus muy extenso

In [ ]:
#Ejecutar la clasificación

#df_final['label'] = df_final['text'].apply(classify_nivele_few_shot)

results = []
for index, row in df_final.iterrows():
  pred = safe_call(classify_nivele_few_shot, row['text'])
  if pred:
    if pred == "a1":
      pred = "A1"
    elif pred == "a2":
      pred = "A2"
    elif pred == "b1":
      pred = "B1"
    elif pred == "b2":
      pred = "B2"
    elif pred == "c1":
      pred = "C1"
    else:
      pred = "error"

    results.append({"label": pred})

# Creamos el dataframe con TODAS las columnas
df_results = pd.DataFrame(results)
df_final = pd.concat([df_final, df_results], axis=1)

In [ ]:
df_final.tail(60)

In [ ]:
df_final[["text", "label"]].iloc[114]

In [ ]:
# Guardar resultados de predicciones sobre el dataset de test / RUN 2
submission = df_final[["label"]]
submission.to_csv("/content/drive/MyDrive/results/run_2/submission.csv", index="id")

In [ ]:
# Guardar resultados de predicciones sobre el dataset de test / RUN 3
submission = df_final[["label"]]
submission.to_csv("/content/drive/MyDrive/results/run_3/submission.csv", index="id")

In [ ]:
# Guardar resultados de predicciones sobre el dataset de test / RUN 4
submission = df_final[["label"]]
submission.to_csv("/content/drive/MyDrive/results/run_4/submission.csv", index="id")

# Inferencia para Nuevos Textos

In [ ]:
def predict_new_text(input_string):
    prediction = classify_nivele_few_shot(input_string)
    print(f"Texto: {input_string[:50]}...")
    print(f"Nivel de lengua: **{prediction.upper()}**")
    return prediction

In [ ]:
# Ejemplo de uso:
nuevo_texto = df_final.iloc[114]['text']
predict_new_text(nuevo_texto)

In [ ]:
# Ejemplo de uso:
nuevo_texto = df_final.iloc[125]['text']
predict_new_text(nuevo_texto)

In [ ]:
# Ejemplo de uso:
nuevo_texto = df_final.iloc[157]['text']
predict_new_text(nuevo_texto)

In [ ]:
# Ejemplo de uso:
nuevo_texto = df_final.iloc[182]['text']
predict_new_text(nuevo_texto)

In [ ]:
# Ejemplo de uso:
nuevo_texto = df_final.iloc[188]['text']
predict_new_text(nuevo_texto)

In [ ]:
# Ejemplo de uso:
nuevo_texto = df_final.iloc[203]['text']
predict_new_text(nuevo_texto)

In [ ]:
# Ejemplo de uso:
nuevo_texto = df_final.iloc[244]['text']
predict_new_text(nuevo_texto)